# 07 — Legacy v3 Descriptor Data Import

기존 `Resume_GPU4070_v2` / `TR3970X_Dual3090_v3` run에서 이미 계산한 descriptor를 **버리지 않고 cumulative ML dataset에 추가**합니다.

- Descriptor representation(PCA/latent state)에는 전부 사용됩니다.
- `v3_seed_based` 구조는 random seed가 연속적인 inverse-design variable이 아니므로, 기본적으로 **Model 1 forward fit에서는 제외**하고 Model 2/representation 학습에는 활용합니다.
- 필요하면 Notebook 01의 `INCLUDE_LEGACY_SEED_ROWS_IN_FORWARD_MODEL=True`로 바꿀 수 있습니다.


In [ ]:
from pathlib import Path
import sys,json,pandas as pd,numpy as np,hashlib
BASE_DIR=Path(r"C:\Users\Administrator\Desktop\Minkyeom\AI-Voxel\Voxel generation")
PROJECT_POINTER=BASE_DIR/".ai_voxel_ml_project.json"
CODE_DIR=Path.cwd()/"Code" if (Path.cwd()/"Code").exists() else Path.cwd();sys.path.insert(0,str(CODE_DIR)) if str(CODE_DIR) not in sys.path else None
# Earlier result formats are both searched. Remove/add entries if you want to restrict the import set.
LEGACY_MASTER_FILES = sorted(BASE_DIR.glob("Result_*/05_Tables/Pool_Structural_Descriptors.csv"))
LEGACY_MASTER_FILES += sorted(BASE_DIR.glob("Voxel_Run_*/03_tables/Pool_Structural_Descriptors.csv"))
LEGACY_FIDELITY='screening'
print('Legacy descriptor master files found:',len(LEGACY_MASTER_FILES))
for _p in LEGACY_MASTER_FILES[:20]: print(' -',_p)


In [ ]:
from voxel_ml_common import load_contract,atomic_csv,atomic_json
c=load_contract(PROJECT_POINTER);master_path=Path(c['all_structures_csv']);old=pd.read_csv(master_path);rows=[]
for p in LEGACY_MASTER_FILES:
    if not p.exists():
        print('[skip missing]',p);continue
    d=pd.read_csv(p) if p.suffix.lower()=='.csv' else pd.read_excel(p,sheet_name='Descriptors' if p.suffix.lower() in {'.xlsx','.xls'} else 0)
    if 'sample_id' not in d:raise ValueError(f'{p} has no sample_id')
    if 'meta__generator_version' not in d:d['meta__generator_version']='v3_seed_based'
    if 'meta__sampling_method' not in d:d['meta__sampling_method']='legacy_import'
    if 'meta__fidelity' not in d:d['meta__fidelity']=d.get('gen__stage',LEGACY_FIDELITY).astype(str) if hasattr(d.get('gen__stage',None),'astype') else LEGACY_FIDELITY
    if 'meta__run_id' not in d:d['meta__run_id']=p.parent.parent.name
    if 'meta__design_id' not in d:d['meta__design_id']=['LEG_'+hashlib.sha256((str(p)+'|'+str(x)).encode()).hexdigest()[:14] for x in d['sample_id'].astype(str)]
    if 'meta__realization_id' not in d:d['meta__realization_id']=d['meta__design_id'].astype(str)+'_R1'
    for j in range(1,17):
        col=f'latent__z{j:02d}'
        if col not in d:d[col]=np.nan
    rows.append(d);print('loaded',len(d),p)
if not rows:raise RuntimeError('No legacy files loaded')
new=pd.concat(rows,ignore_index=True,sort=False);combined=pd.concat([old,new],ignore_index=True,sort=False)
keys=[x for x in ['meta__run_id','sample_id','meta__fidelity'] if x in combined]
if keys:combined=combined.drop_duplicates(keys,keep='last')
atomic_csv(combined,master_path);atomic_csv(new,Path(c['data_root'])/'legacy_import_latest.csv')
print('Cumulative rows:',len(old),'->',len(combined));display(new.head())
